# Kỹ thuật Dropout(Bỏ học) trong Deep Learning

## 1. Lý thuyết

### 1.1 Dropout trong mạng Neural là gì?

Hiểu một cách đơn giản thì dropout là việc bỏ qua các đơn vị (tức là 1 nút mạng) trong quá trình đào tạo 1 cách ngẫu nhiên. Bằng việc bỏ qua này thì đơn vị đó sẽ không được xem xét trong quá trình forward và backward. theo đó, p được gọi là xác suất giữ lại 1 nút mạng trong mỗi giai đoạn huấn luyện, vì thế xãc xuất nó bị loại bỏ là (1-p).

### 1.2 tại sao lại cần Dropout

Câu hỏi là: tại sao phải tắt 1 số nút mạng theo đúng gnhiax đen trong quá trình train?

$\rightarrow$ Tránh học tủ (Over-fitting)

Nếu 1 lớp fully connected có quá nhiều tham số và chiếm hầu hết tham số, các nút mạng trong lớp đó quá phụ thuộc lẫn nhau trong quá trình huấn luyện thì sẽ hạn chế sức mạnh của mỗi nút, dẫn đến việc kết hợp quá mức.

### 1.3 Các kỹ thuật khác

Nếu bạn mốn biết droput là gì, thì chỉ 2 phần lý thuyết phía trên là đủ. Ở phần này mình ới thiệu 1 số kỹ thuật có cùng tác dụng với dropout.

Trong machine learning, việc chhs quy hoá (regularization) sẽ làm giảm over-fitting bằng cách thêm 1 khoảng giá trị 'phạt' vào hàm loss. Bằng cách thêm 1 giá trị như vậy. mô hình của bạn sẽ không học quá nhiều sự phụ thuộc giữa các trọng số. Chắc hẳn nhiều người đã biết đén logistic regression thì đều biết đến L1 (Laplacian) và L2 (Gaussian) là 2 kỹ thuật 'phạt'.

- Qúa trình training: đối với mỗi lớp ẩn, mỗi example, mỗi vòng lặp, ta sẽ bỏ học 1 cách ngẫy nhiên với xác suất (1-p) cho mỗi nút mạng.
- Quá trình test: Sử dụng tât cả các kích hoạt, nhưng sẽ giảm đi 1 hệ số p (để tính cho các kích hoạt bị bỏ học)

![](image1.png)

### 1.4 Một số nhận xét

- Dropout sẽ được học thêm các tính năng mạnh mẽ hữu ích.
- Nó gần như tăng gấp đôi số epochs cần thiết để hội tụ. Tuy nhiên, thời gian cho mỗi epoch là ít hơn.
- Ta có H đơnvị ẩn, với xác suất bỏ học cho mỗi đơn vị là (1-p) thì ta có thể có 2^H mô hình có thể có. Nhưng trong giai đoạn test, tất cả các nút mạng phải được xét đến, và mỗi activation sẽ gaimr đi 1 hệ số p.

## 2. Thực hành

Đặt vấn đề: Bạn đi xem 1 trận đấu bóng đá và bạn thử dự đoán xem thủ môn sút vào vị trí nào thì cầu thủ nhà đánh đầu được quả bóng.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from reg_utils import sigmoid, relu, plot_decision_boundary, initialize_parameters, load_2D_dataset, predict_dec
from reg_utils import compute_cost, predict, forward_propagation, backward_propagation, update_parameters
import sklearn
import sklearn.datasets
import scipy.io
from testCases import *

%matplotlib inline
plt.rcParams['figure.figsize'] = (7.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

In [ ]:
train_X, train_Y, test_X, test_Y = load_2D_dataset()

Ta được kết quả:

![](image2.png)

- Dấu chấm đỏ là cầu thủ nhà đã từng đánh đầu, chấm xanh là cầu thủ bạn đánh đầu. Việc chúng ta là dự đoán xem thủ môn nên sút bóng vào khu vực nào để cầu thủ nhà đánh đầu được. Nhìn có vẽ như chỉ cần kẻ 1 được thẳng để phân chia 2 khu vực là được.

## 2.1 Mô hình không có chính quy hoá

In [ ]:
def model(
        X,
        Y,
        learning_rate=0.3,
        num_iterations=30000,
        print_cost=True
):
    """
    Triển khai mạng với 3 layer: LINEAR->RELU->LINEAR->RELU->LINEAR->SIGMOID.

    :param X: dữ liệu đầu vào, kích thức (input size, number of examples)
    :param Y: 1 vector (1 là chấm xanh / 0 là chấm đỏ), kích thước (output size, number of examples)
    :param learning_rate: tỷ lệ học
    :param num_iterations: số epochs
    :param print_cost: nếu là True, in ra cos cho mỗi 10000 vòng lặp
    :return: tham số học được, dùng để dự đoán
    """


    grad = {}
    costs = []
    m = X.shape[1]
    layer_dims = [X.shape[0], 20, 3, 1]


    # Initialize parameters dictionary
    parameters = initialize_parameters(layer_dims)

    # Loop gradient
    for i in range(0, num_iterations):

        # Forward propagation: LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
        a3, cache = forward_propagation(X, parameters)

        # Cost function
        cost = compute_cost(a3, Y)

        grads = backward_propagation(X, Y, cache)

        # Print the loss every 10000 iterations
        if print_cost and i % 10000 == 0:
            print("Cost after iteration {}: {}".format(i, cost))
        if print_cost and i % 1000 == 0:
            costs.append(cost)


    # plot the cost
    plt.plot(costs)
    plt.ylabel('cost')
    plt.xlabel('iterations (x1,000)')
    plt.title("Learning rate =" + str(learning_rate))
    plt.show()

    return parameters

Hàm dự đoán

In [ ]:
predictions_train = predict(train_X, train_Y, parameters)
predictions_test = predict(test_X, test_Y, parameters)

Xem kết quả

Cost after iteration 0: 0.6557412523481002

Cost after iteration 10000: 0.16329987525724216

Cost after iteration 20000: 0.13851642423255986

...

On the training set:

Accuracy: 0.947867298578

On the test set:

Accuracy: 0.915

![](image3.png)

Có thể thấy độ chính xác ở tập training là 94% và tập test là 91% (khá cao). Ta sẽ visualize 1 chút

![](image4.png)

Khi không có chính quy hóa, ta thấy đường phân chia vẽ rất chi tiết, tức là nó đang over-fitting.

## 2.2 Mô hình chính quy hóa với Dropout

### 2.2.1 Quá trình Forward Propagation

In [ ]:
def forwar_propagation_with_dropout(
        X,
        parameters,
        keep_prob=0.5
):
    """
    Triển khai 3 layer: LINEAR -> RELU + DROPOUT -> LINEAR -> RELU + DROPOUT -> LINEAR -> SIGMOID.

    :param X: Dữ liệu đầu vào, kích thước (2, number of examples)
    :param parameters: Các đối số chúng ta có "W1", "b1", "W2", "b2", "W3", "b3":
                            W1 -- weight matrix of shape (20, 2)
                            b1 -- bias vector of shape (20, 1)
                            W2 -- weight matrix of shape (3, 20)
                            b2 -- bias vector of shape (3, 1)
                            W3 -- weight matrix of shape (1, 3)
                            b3 -- bias vector of shape (1, 1)
    :param keep_prob: xác suất giữ lại 1 unit
    :return:
        A3 -- giá trị đầu ra mô hình, kích thước (1,1)
        cache -- lưu các đối số để tính cho phần Backward Propagation
    """


    np.random.seed(1)

    # Retrieve Parameters
    W1 = parameters["W1"]
    b1 = parameters["b1"]
    W2 = parameters["W2"]
    b2 = parameters["b2"]
    W3 = parameters["W3"]
    b3 = parameters["b3"]


    # LINEAR -> RELU -> LINEAR -> RELU -> LINEAR -> SIGMOID
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)

    # Step 1: khởi tạo ngẫu nhiên 1 ma trận kích thước bằng kích thước A1, giá trị (0, 1)
    D1 = np.random.rand(
        A1.shape[0],
        A1.shape[1]
    )
    # Step 2: chuyển các giá trị về 0 hoặc 1, trả về 1 nếu giá trị đó nhỏ hơn keep_prob
    D1 = D1 < keep_prob
    # Step 3: giữ nguyên các phần tự trong A1 ứng với phần tử 1 của D1, và đổi thành 0 nếu vị trị trong D1 tương tứng là 0
    A1 = A1 * D1
    # Step 4: giảm đi 1 hệ số keep_prob, để tính cho các phần tử đã bỏ học.
    A1 = A1 / keep_prob


    D2 = np.random.rand(
        A2.shape[0],
        A2.shape[1]
    )
    D2 = D2 < keep_prob
    A2 = A2 * D2
    A2 = A2 / keep_prob



    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)


    cache = (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3)


    return A3, cache

### 2.2.2. Quá trình Backward Propagation

In [ ]:
def backward_propagation_with_dropout(
        X,
        Y,
        cache,
        keep_prob
):
    """

    :param X: Dữ liệu đầu vào, kích thước (2, number of examples)
    :param Y: kích thước (output size, number of examples)
    :param cache: lưu đầu ra của forward_propagation_with_dropout()
    :param keep_prob: như forward
    :return: gradients -- Đạo hàm của tất cả các weight, activation
    """

    m = X.shape[1]

    (Z1, D1, A1, W1, b1, Z2, D2, A2, W2, b2, Z3, A3, W3, b3) = cache

    dZ3 = A3 - Y
    dW3 = 1. / m * np.dot(dZ3, A2.T)
    db3 = 1. / m * np.sum(dZ3, axis=1, keepdims=True)
    dA2 = np.dot(W3.T, dZ3)

    # Step 1: Áp dụng D2 để tắt các unit tương ứng với forward
    dA2 = dA2 * D2
    # Step 2: Giảm giá trị 1 hệ số keep_prob
    dA2 = dA2 / keep_prob


    dZ2 = np.multiply(dA2, np.int64(A2 > 0))
    dW2 = 1. / m * np.dot(dZ2, A1.T)
    db2 = 1. / m * np.sum(dZ2, axis=1, keepdims=True)


    dA1 = np.dot(W2.T, dZ2)

    dA1 = dA1 * D1
    dA1  = dA1 / keep_prob

    dZ1 = np.multiply(dA1, np.int64(A1 > 0))
    dW1 = 1. / m * np.dot(dZ1, X.T)
    db1 = 1. / m * np.sum(dZ1, axis=1, keepdims=True)

    gradients = {
        "dZ3": dZ3, "dW3": dW3, "db3": db3,"dA2": dA2,
         "dZ2": dZ2, "dW2": dW2, "db2": db2, "dA1": dA1,
         "dZ1": dZ1, "dW1": dW1, "db1": db1
    }

    return  gradients

Sau khi có Forward và Backward, ta thay 2 hàm này vào hàm model của phần trước:

In [ ]:
parameters = model(
    train_X,
    train_Y,
    keep_prob=0.86,
    learning_rate=0.3
)

predictions_train = predict(train_X, train_Y, parameters)
predictions_test = predict(test_X, test_Y, parameters)

Kết quả:

Cost after iteration 10000: 0.06101698657490559

Cost after iteration 20000: 0.060582435798513114

...

On the train set:

Accuracy: 0.928909952607

On the test set:

Accuracy: 0.95

![](image5.png)

Ta thấy, độ chính xác trong tập test đã lên đến 95%, mặc dù tập training bị giảm. Thực hiện visualize:

In [ ]:
plt.title("Model with dropout")
axes = plt.gca()
axes.set_xlim([-0.75, 0.40])
axes.set_ylim([-0.75, 0.65])
plot_decision_boundary(lambda x: predict_dec(parameters, x.T), train_X, train_Y)

![](image6.png)

## 2.3 Chú ý

- Không dùng Dropout cho quá trình test
- Áp dụng Dropout cho cả quá trình Forward và Backward
- Giá trị kích hoạt phải giảm đi 1 hệ số keep_prob, tính cả cho những nút bỏ học.

# Accuracy train nói lên điều gì?

- Cho biết model học tốt dữ liệu nó đã thấy hay chưa
- Accuracy train cao có thể chỉ đơn giản là:: Model nhớ (memorize) dữ liệu, Overfitting nặng

# Các trường hợp thường gặp:

### Case 1: Train cao – Test cao ✅

Train acc: 95%
Test  acc: 93%

- ✔ Model học đúng pattern
- ✔ Generalization tốt

### Case 2: Train cao – Test thấp ❌ (Overfitting)

Train acc: 99%
Test  acc: 60%

- ✘ Model học thuộc lòng noise
- ✘ Không dùng được ngoài đời

### Case 3: Train thấp – Test thấp ❌ (Underfitting)

Train acc: 60%
Test  acc: 58%

- ✘ Model quá đơn giản
- ✘ Chưa học được pattern

### Case 4: Train thấp – Test cao (hiếm)

Train acc: 70%
Test  acc: 85%

Thường là do:
- Train set khó hơn
- Data leakage
- Chia dataset không chuẩn

# $\rightarrow$ Test accuracy quyết định model có dùng được hay không